In [0]:
dbutils.widgets.removeAll()

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
dbutils.widgets.text("container", "raw")
dbutils.widgets.text("catalogo", "catalog_clinica")
dbutils.widgets.text("esquema", "bronze")
dbutils.widgets.text("storageName", "adlsproyecto2404")

In [0]:
container = dbutils.widgets.get("container")
catalogo = dbutils.widgets.get("catalogo")
esquema = dbutils.widgets.get("esquema")
storageName = dbutils.widgets.get("storageName")

ruta = f"abfss://{container}@{storageName}.dfs.core.windows.net/atenciones.csv"

In [0]:
atenciones_schema = StructType(fields=[
    StructField("id_atencion", IntegerType(), True),
    StructField("id_cita", IntegerType(), True),
    StructField("fecha_atencion", TimestampType(), True),
    StructField("id_paciente", IntegerType(), True),
    StructField("id_medico", IntegerType(), True),
    StructField("diagnostico", StringType(), True),
    StructField("tratamiento", StringType(), True),
    StructField("duracion_min", IntegerType(), True),
    StructField("requiere_seguimiento", StringType(), True)
])

In [0]:
df_atenciones_read = spark.read.option('header', True)\
                               .schema(atenciones_schema)\
                               .csv(ruta)

In [0]:
df_atenciones_ingestion = df_atenciones_read.withColumn("INGESTION_DATE", current_timestamp())

In [0]:
df_atenciones_final = df_atenciones_ingestion.select(
    "id_atencion",
    "id_cita",
    "fecha_atencion",
    "id_paciente",
    "id_medico",
    "diagnostico",
    "tratamiento",
    "duracion_min",
    "requiere_seguimiento",
    "INGESTION_DATE"
)

In [0]:
df_atenciones_final.write.mode("overwrite").insertInto(f"{catalogo}.{esquema}.atenciones")